In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.utils.data import random_split
from torchvision import transforms

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from sklearn.metrics import roc_auc_score

# Tabular

### Loading Tabular Dataset

In [ ]:
mineral_cols = ["AU", "AG", "CU", "CO", "NI"]
target_cols = ["target_AU", "target_AG", "target_CU", "target_CO", "target_NI"]
target_col_string = "target_AU"
target_col = [target_col_string] # For single target prediction

df_processed = pd.read_csv('../processed(500)_geology_ml_training_data.csv', low_memory=False)
df_processed = df_processed[df_processed["hasImage"]] # Only get rows that have images
df_processed = df_processed.dropna(subset=target_col).copy()
df_processed[target_col_string] = (df_processed[target_col_string] > 0).astype(np.float32)

tabular_cols = [col for col in df_processed.columns if col not in  mineral_cols + target_cols + ['spatial_cluster', 'CODE_LITH', 'STRAT', 'Sample_ID', 'hasImage']]

bad_cols = df_processed[tabular_cols].select_dtypes(include=["object"]).columns
print(bad_cols)

df_processed[tabular_cols].head()

Index([], dtype='object')


,Easting,Northing,dist_fault,dist_cont,dist_fault_norm,dist_cont_norm,AU_missing,AG_missing,CU_missing,CO_missing,...,V3,V3A,V3B,V3D,V3F,V3G,V3H,V4,V4A,V4FO
4,-823281.410385,445223.349511,8620.008852,708.354658,0.040018,0.004530,0.000000,0.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
6,-822577.690915,449159.895115,5937.687683,8.809315,0.027565,0.000056,0.333333,0.333333,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
7,-822498.982021,419697.736482,7963.856827,225.416115,0.036971,0.001442,0.000000,1.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
8,-822352.108967,421386.167458,8951.045298,317.167447,0.041554,0.002028,0.000000,1.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
9,-822343.618110,420346.833544,8246.925134,289.456545,0.038286,0.001851,0.000000,0.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


### Splitting Tabular Dataset

In [3]:
# Splitting dataframe
train_size = int(0.7 * len(df_processed))
val_size   = int(0.15 * len(df_processed))
test_size  = len(df_processed) - train_size - val_size

train_df, temp_df = train_test_split(df_processed, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

### Tabular Branch MLP

In [4]:
class TabularBranch(nn.Module):
    def __init__(self, input_dim, embedding_dim=64, dropout=0.30):
        super().__init__()

        self.net = nn.Sequential(
            nn.BatchNorm1d(input_dim),

            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(64, embedding_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

### Creating Tabular Embeddings

In [5]:
tabular_dim = len(tabular_cols)

tabular_branch = TabularBranch(input_dim=tabular_dim, embedding_dim=64, dropout=0.30)

batch_size = 32
x_tabular_torch = torch.randn(batch_size, tabular_dim)

tab_embedding = tabular_branch(x_tabular_torch)

print(tab_embedding.shape)

torch.Size([32, 64])


# Images

### Image Branch ResNet18

In [6]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout=0.10):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.dropout = nn.Dropout2d(dropout)

        self.skip = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.skip(x)

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))

        out = out + identity
        out = F.relu(out)

        return out


class ImageBranch(nn.Module):
    def __init__(self, in_channels=1, embedding_dim=256, dropout=0.20):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        self.layers = nn.Sequential(
            ResidualBlock(32, 32, stride=1, dropout=dropout),
            ResidualBlock(32, 64, stride=2, dropout=dropout),
            ResidualBlock(64, 128, stride=2, dropout=dropout),
            ResidualBlock(128, 256, stride=2, dropout=dropout)
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layers(x)
        x = self.pool(x)
        x = self.projection(x)

        return x

### Creating Image Embeddings

In [7]:
image_branch = ImageBranch(in_channels=3, embedding_dim=256, dropout=0.20)

batch_size = 32
images = torch.randn(batch_size, 3, 200, 200)

image_embedding = image_branch(images)

print(image_embedding.shape)

torch.Size([32, 256])


# Model

### Fusion Model of Tabular and Image data

In [8]:
class GeoMineralModel(nn.Module):
    def __init__(self, image_branch, tabular_branch):
        super().__init__()

        self.image_branch = image_branch
        self.tabular_branch = tabular_branch

        self.head = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)   # 1 if only one target. For five targets (AU AG CU CO NI) would be nn.Linear(128, 5)
        )

    def forward(self, image, tabular):
        image_emb = self.image_branch(image)
        tabular_emb = self.tabular_branch(tabular)

        x = torch.cat([image_emb, tabular_emb], dim=1)

        return self.head(x)

### Dataset

In [9]:
class GeologicalDataset(Dataset):
    def __init__(self, dataframe, images_path, tabular_cols, target_cols, transform=None):
        self.df = dataframe
        self.images_path = images_path
        self.tabular_cols = tabular_cols
        self.target_cols = target_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_id = row["spatial_cluster"]
        image = Image.open(f"{self.images_path}{image_id}_native_res.jpg").convert("RGB")

        if self.transform:
            image = self.transform(image)

        # tabular = torch.tensor(self.df.iloc[idx][self.tabular_cols].values, dtype=torch.float32)
        tabular = torch.from_numpy(row[self.tabular_cols].to_numpy(dtype=np.float32))

        # target = torch.tensor(self.df.iloc[idx][self.target_cols].values, dtype=torch.float32)
        target = torch.from_numpy(row[self.target_cols].to_numpy(dtype=np.float32))

        return image, tabular, target

### Creating Datasets and DataLoaders

In [10]:
images_path = 'images_cluster500/'
image_transform = transforms.Compose([transforms.Resize((200, 200)), transforms.ToTensor()])

# Creating datasets
train_dataset = GeologicalDataset(train_df, images_path, tabular_cols, target_col, transform=image_transform)
val_dataset = GeologicalDataset(val_df, images_path, tabular_cols, target_col, transform=image_transform)
test_dataset = GeologicalDataset(test_df, images_path, tabular_cols, target_col, transform=image_transform)

# Creating loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

### Creating Model, Optimizer and Criterion

In [11]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
if torch.backends.mps.is_available():
  print("MPS available")
else:
  print("MPS not available")
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GeoMineralModel(image_branch,tabular_branch).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4) # Why AdamW?

# Setting actual weights from the training set for the imbalanced classes
pos_weights = []
for col in target_col: # Using just our main target
    positives = train_df[col].sum()
    negatives = len(train_df) - positives
    pos_weights.append(negatives / max(positives, 1))
pos_weight = torch.tensor(pos_weights, dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

MPS available


## Training Model

In [12]:
model_path = f'best_model.pth'

# TRAINING
num_epochs = 20
best_val_loss = float("inf")
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    model.train()
    train_loss = 0
    for images, tabular, targets in train_loader:
        images = images.to(device)
        tabular = tabular.to(device)
        targets = targets.to(device)

        logits = model(images, tabular)
        loss = criterion(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for images, tabular, targets in val_loader:
            images = images.to(device)
            tabular = tabular.to(device)
            targets = targets.to(device)

            logits = model(images, tabular)
            loss = criterion(logits, targets)

            val_loss += loss.item()

    if val_loss < best_val_loss:
      best_val_loss = val_loss
      torch.save(model.state_dict(), model_path)

    print(f"Epoch {epoch+1} | "
          f"Train Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f}")

Epoch 1/20
Epoch 1 | Train Loss: 1.3254 | Val Loss: 1.3902
Epoch 2/20
Epoch 2 | Train Loss: 1.3235 | Val Loss: 1.3224
Epoch 3/20
Epoch 3 | Train Loss: 1.2279 | Val Loss: 1.2982
Epoch 4/20
Epoch 4 | Train Loss: 1.1214 | Val Loss: 1.2608
Epoch 5/20
Epoch 5 | Train Loss: 1.0446 | Val Loss: 1.3108
Epoch 6/20
Epoch 6 | Train Loss: 1.0026 | Val Loss: 1.2000
Epoch 7/20
Epoch 7 | Train Loss: 0.9694 | Val Loss: 1.3051
Epoch 8/20
Epoch 8 | Train Loss: 0.9425 | Val Loss: 1.2724
Epoch 9/20
Epoch 9 | Train Loss: 0.9337 | Val Loss: 1.3724
Epoch 10/20
Epoch 10 | Train Loss: 0.8877 | Val Loss: 1.3679
Epoch 11/20
Epoch 11 | Train Loss: 0.8886 | Val Loss: 1.2738
Epoch 12/20
Epoch 12 | Train Loss: 0.8427 | Val Loss: 1.3604
Epoch 13/20
Epoch 13 | Train Loss: 0.8396 | Val Loss: 1.3581
Epoch 14/20
Epoch 14 | Train Loss: 0.8314 | Val Loss: 1.4937
Epoch 15/20
Epoch 15 | Train Loss: 0.8156 | Val Loss: 1.6700
Epoch 16/20
Epoch 16 | Train Loss: 0.7985 | Val Loss: 1.8337
Epoch 17/20
Epoch 17 | Train Loss: 0.9379 

In [ ]:
print(f"Val Loss: {best_val_loss/len(val_loader):.4f}")

Val Loss: 1.2000


## Testing Model

In [15]:
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

all_targets = []
all_probs = []

with torch.no_grad():
    for images, tabular, targets in test_loader:

        images = images.to(device)
        tabular = tabular.to(device)

        logits = model(images, tabular)

        probs = torch.sigmoid(logits)

        all_targets.append(targets.numpy())
        all_probs.append(probs.cpu().numpy())

y_true = np.concatenate(all_targets)
y_prob = np.concatenate(all_probs)

print("ROC AUC:", roc_auc_score(y_true, y_prob))
print("PR AUC:", average_precision_score(y_true, y_prob))
print("Positive rate:", y_true.mean())

ROC AUC: 0.791714461771816
PR AUC: 0.1487369869583721
Positive rate: 0.03311258
